In [1]:
import pandas as pd

nav = pd.read_csv("E:/bluestock_mf_capstone/data/raw/02_nav_history.csv")

nav.head()

,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [2]:
print(nav.info())
print(nav.columns.tolist())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   amfi_code  46000 non-null  int64  
 1   date       46000 non-null  object 
 2   nav        46000 non-null  float64
dtypes: float64(1), int64(1), object(1)
memory usage: 1.1+ MB
None
['amfi_code', 'date', 'nav']


In [3]:
nav['date'] = pd.to_datetime(
    nav['date'],
    errors='coerce'
)

In [4]:
nav = nav.sort_values(
    ['amfi_code','date']
)

In [5]:
nav.isnull().sum()

amfi_code    0
date         0
nav          0
dtype: int64

In [6]:
nav['nav'] = nav.groupby(
    'amfi_code'
)['nav'].ffill()

In [7]:
nav = nav.drop_duplicates()

In [8]:
nav = nav[nav['nav'] > 0]

In [9]:
nav.to_csv(
    "../data/processed/nav_history_clean.csv",
    index=False
)

TASK 2: Clean investor_transactions.csv

In [16]:
import pandas as pd

txn = pd.read_csv("../data/raw/08_investor_transactions.csv")

print("Shape:", txn.shape)
txn.head()

Shape: (32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [17]:
txn["transaction_date"] = pd.to_datetime(
    txn["transaction_date"],
    errors="coerce"
)

In [18]:
txn['transaction_type'].unique()

array(['SIP', 'Redemption', 'Lumpsum'], dtype=object)

In [19]:
txn['transaction_type'] = (
    txn['transaction_type']
    .str.strip()
    .str.title()
)

In [20]:
valid_types = [
    "Sip",
    "Lumpsum",
    "Redemption"
]

txn = txn[
    txn["transaction_type"].isin(valid_types)
]

In [21]:
txn = txn[
    txn["amount_inr"] > 0
]

In [22]:
print("KYC Status Values:")
print(txn["kyc_status"].value_counts())

# Keep only valid KYC values
valid_kyc = [
    "Verified",
    "Pending",
    "Rejected"
]

txn = txn[
    txn["kyc_status"].isin(valid_kyc)
]

KYC Status Values:
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64


In [23]:
print("\nFinal Shape:")
print(txn.shape)

print("\nTransaction Types:")
print(txn["transaction_type"].unique())

print("\nKYC Status:")
print(txn["kyc_status"].unique())


Final Shape:
(32778, 13)

Transaction Types:
['Sip' 'Redemption' 'Lumpsum']

KYC Status:
['Verified' 'Pending']


In [24]:
txn.to_csv(
    "../data/processed/investor_transactions_clean.csv",
    index=False
)

print("\nCleaned file saved successfully!")


Cleaned file saved successfully!


Code for Task 3

In [26]:
import pandas as pd

# Load dataset
perf = pd.read_csv("../data/raw/07_scheme_performance.csv")

print("Original Shape:", perf.shape)

Original Shape: (40, 19)


In [27]:
print(perf.columns)

Index(['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan',
       'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct',
       'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
       'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct',
       'morningstar_rating', 'risk_grade'],
      dtype='object')


In [28]:
return_cols = [
    "return_1yr_pct",
    "return_3yr_pct",
    "return_5yr_pct",
    "benchmark_3yr_pct"
]

for col in return_cols:
    perf[col] = pd.to_numeric(
        perf[col],
        errors="coerce"
    )

In [29]:
print(
    perf[return_cols].isnull().sum()
)

return_1yr_pct       0
return_3yr_pct       0
return_5yr_pct       0
benchmark_3yr_pct    0
dtype: int64


In [30]:
perf = perf.dropna(
    subset=return_cols
)

In [31]:
perf["negative_sharpe_flag"] = (
    perf["sharpe_ratio"] < 0
)

In [32]:
print(
    "Negative Sharpe Ratios:",
    perf["negative_sharpe_flag"].sum()
)

Negative Sharpe Ratios: 0


In [33]:
negative_sharpe = perf[
    perf["negative_sharpe_flag"] == True
]

print(negative_sharpe[
    ["scheme_name", "sharpe_ratio"]
].head())

Empty DataFrame
Columns: [scheme_name, sharpe_ratio]
Index: []


In [34]:
invalid_expense = perf[
    ~perf["expense_ratio_pct"].between(
        0.1,
        2.5
    )
]

print(
    "Invalid Expense Ratios:",
    invalid_expense.shape[0]
)

Invalid Expense Ratios: 0


In [35]:
perf = perf[
    perf["expense_ratio_pct"].between(
        0.1,
        2.5
    )
]

In [36]:
print("\nFinal Shape:")
print(perf.shape)

print("\nExpense Ratio Summary:")
print(
    perf["expense_ratio_pct"].describe()
)

print("\nSharpe Ratio Summary:")
print(
    perf["sharpe_ratio"].describe()
)


Final Shape:
(40, 20)

Expense Ratio Summary:
count    40.000000
mean      1.237000
std       0.386584
min       0.550000
25%       0.787500
50%       1.425000
75%       1.540000
max       1.640000
Name: expense_ratio_pct, dtype: float64

Sharpe Ratio Summary:
count    40.000000
mean      1.361750
std       1.475805
min       0.800000
25%       0.865000
50%       0.925000
75%       0.985000
max       7.680000
Name: sharpe_ratio, dtype: float64


In [37]:
perf.to_csv(
    "../data/processed/scheme_performance_clean.csv",
    index=False
)

print(
    "scheme_performance_clean.csv saved successfully!"
)

scheme_performance_clean.csv saved successfully!
